<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Phase40_Google_Colab_Drive_Auto_Fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 · Phase 40 — Google Drive auto-discovery, source evidence gates & Arm A mapping contract

**Run All in Google Colab. No manual files.upload() and no downloading or uploading inputs.**

This is the Drive-first revision of Phase 40. It mounts your Google Drive (one authorization in Colab), searches **My Drive** and accessible **Shared drives** for the **eight Phase 39 artifacts**, tolerates downloaded duplicate suffixes such as `(1)`, and selects inputs only when their bytes match the hash manifest from **the Phase 39 run supplied for this project**. Matching files can be in different folders. It copies only these validated synthetic artifacts into temporary Colab space so SQLite runs on a local disk. Phase 40 also builds its CSV/JSON/ZIP outputs locally, then copies and checksum-verifies the final outputs on Drive (avoiding first-write Drive filesystem failures in code cell 5). If any expected file is missing or changed, it stops with a specific missing/hash diagnostic instead of asking for an upload or substituting another run.

Phase 40 independently checks synthetic source-ledger integrity, reconciles Phase 39 artifacts, creates a **logical proposed mapping contract**, and records outstanding Arm A evidence gates. **Not** an official OMOP/MI-CDM ETL, not a real-MRI run, and not an MCI clinical cohort. No private Athena exports, concept decisions, raw participant-level data, or Drive paths are placed in its public-safe outputs.

**Where outputs go:** by default, `MyDrive/ICHI2027/Phase40/run_<UTC timestamp>/` (CSV, JSON, README and ZIP), created automatically. If you have an independently approved *private evidence receipt* on Drive, the notebook can find it without copying it to the public output. Approvals never substitute for executing official target validation.

**Optional settings in the next cell:** `DRIVE_FOLDER_HINT` to speed up search in a known folder (otherwise search all Drive), and `PINNED_PHASE39_MANIFEST_SHA256` to verify that the manifest is exactly the one supplied with Phase 39. On non-Colab Python, set `PHASE40_LOCAL_DRIVE_ROOT` for a local test.


In [1]:
# CODE CELL 1/8 — Mount Drive and prepare local working directory
from pathlib import Path
import os, csv, json, hashlib, sqlite3, datetime, re, shutil, tempfile
from collections import Counter, defaultdict

# OPTIONAL: set a known folder relative to MyDrive, such as 'Research/ICHI2027'.
# Empty means automatically search all of MyDrive plus accessible Shared drives.
DRIVE_FOLDER_HINT = os.environ.get('PHASE40_DRIVE_FOLDER_HINT', '').strip()

# Pin to the exact Phase 39 SHAREABLE.json uploaded with this notebook.
# Deliberately fail if Drive contains a different Phase 39 experiment/run.
PINNED_PHASE39_MANIFEST_SHA256 = os.environ.get(
    'PHASE40_PINNED_MANIFEST_SHA256',
    '381734533579c739336b3d99dd221cf2de5b37b174743b7cbf7bfdc50201f0c8'
).lower()

IN_COLAB = False
if os.environ.get('PHASE40_LOCAL_DRIVE_ROOT'):
    # Reproducibility/test path; Colab users need not set this.
    DRIVE_ROOTS = [Path(os.environ['PHASE40_LOCAL_DRIVE_ROOT']).expanduser().resolve()]
    MYDRIVE = DRIVE_ROOTS[0]
    print('Local Drive-mirror test mode:', MYDRIVE)
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        IN_COLAB = True
        MYDRIVE = Path('/content/drive/MyDrive')
        if not MYDRIVE.is_dir():
            raise RuntimeError('Drive mounted but MyDrive is missing; check your Google account access.')
        DRIVE_ROOTS = [MYDRIVE]
        shared = Path('/content/drive/Shareddrives')
        if shared.is_dir(): DRIVE_ROOTS.append(shared)
        print('Google Drive is ready. Search roots:', *map(str, DRIVE_ROOTS), sep='\n  ')
    except ImportError as e:
        raise RuntimeError('Run in Google Colab, or set PHASE40_LOCAL_DRIVE_ROOT for local execution.') from e

if DRIVE_FOLDER_HINT:
    hinted = Path(DRIVE_FOLDER_HINT).expanduser()
    if not hinted.is_absolute(): hinted = MYDRIVE / hinted
    if not hinted.is_dir():
        raise FileNotFoundError('DRIVE_FOLDER_HINT does not exist: ' + str(hinted))
    DRIVE_ROOTS = [hinted.resolve()]
    print('Search narrowed to:', hinted)

# Final output is persistent Drive storage; intermediate files are built locally and copied at the end.
base_out = Path(os.environ.get('PHASE40_OUTPUT_DIR', str(MYDRIVE / 'ICHI2027' / 'Phase40')))
run_tag = datetime.datetime.now(datetime.timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')
OUT = base_out / run_tag
OUT.mkdir(parents=True, exist_ok=False)
WORK = Path(tempfile.mkdtemp(prefix='ichi_phase40_'))
IN = WORK / 'verified_phase39'
IN.mkdir()
STAGE_OUT = WORK / 'phase40_results'
STAGE_OUT.mkdir()
print('Output folder on Drive:', OUT)
print('Intermediate CSV/JSON/ZIP files will be built on local Colab disk first.')


Mounted at /content/drive
Google Drive is ready. Search roots:
  /content/drive/MyDrive
Output folder on Drive: /content/drive/MyDrive/ICHI2027/Phase40/run_20260929T054826_100243Z
Intermediate CSV/JSON/ZIP files will be built on local Colab disk first.


## 1 · Automatically discover the Phase 39 artifacts on Drive and verify every SHA-256

The search is **filename-only** until the required Phase 39 manifest is identified. It hashes only likely files. Duplicate download names such as `ICHI2027_Phase39_SOURCE_LEDGER_SCHEMA(1).sql` and `... (2).sql` are supported. If two copies have the same approved content, one is chosen deterministically; a different-content copy is rejected. No input can silently come from a different experiment, because the Phase 39 manifest hash and all seven companion hashes must match.

If nothing is found, the notebook raises a meaningful `FileNotFoundError` listing missing artifact names. If you have files in an unusually named folder, set `DRIVE_FOLDER_HINT` in the prior cell. This notebook **does not** call `files.upload()`.


In [2]:
# CODE CELL 2/8 — Find and hash-verify all eight Phase 39 artifacts
BASE = 'ICHI2027_Phase39_'
REQUIRED = ['SHAREABLE.json','SQL_STAGING_COUNTS.csv','SOURCE_SQL_RECONCILIATION.csv',
            'SOURCE_COHORT_AGREEMENT.csv','NEGATIVE_SQL_TESTS.csv','ARM_A_READINESS.csv',
            'SOURCE_LEDGER_SCHEMA.sql','SYNTHETIC_SOURCE_LEDGER.sqlite']
EXPECTED_PHASE = '39_sql_source_ledger_cohort_replay_and_integrity'

def sha(p):
    h = hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda:f.read(2 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

# One recursive directory walk, with exact filename matching (and optional download suffixes).
file_patterns = {
    s: re.compile(r'^'+re.escape(BASE+s.rsplit('.',1)[0])+
                  r'(?:\s*\(\d+\))?\.'+re.escape(s.rsplit('.',1)[1])+r'$', re.I)
    for s in REQUIRED
}
candidates = {s: [] for s in REQUIRED}
private_receipts = []
SKIP_DIRS = {'.git','node_modules','.ipynb_checkpoints','.Trash-1000'}
for root in DRIVE_ROOTS:
    if not root.is_dir():
        continue
    for current, dirs, files_here in os.walk(root, topdown=True, followlinks=False):
        dirs[:] = [d for d in dirs if d not in SKIP_DIRS and not d.startswith('.Trash')]
        for fname in files_here:
            if fname == 'phase40_private_approvals.json':
                private_receipts.append(Path(current)/fname)
            for suffix, pattern in file_patterns.items():
                if pattern.fullmatch(fname):
                    candidates[suffix].append(Path(current)/fname)
                    break
print('Drive matches by expected artifact:', {s:len(paths) for s,paths in candidates.items()})

if not candidates['SHAREABLE.json']:
    raise FileNotFoundError('No Phase 39 SHAREABLE.json found on your searched Drive roots. '
                            'The notebook cannot recover files that are not actually on Drive.')

# All candidate manifests are checked against the pinned hash; never silently pick another run.
manifest_choices = []
for p in candidates['SHAREABLE.json']:
    digest = sha(p)
    if PINNED_PHASE39_MANIFEST_SHA256 and digest != PINNED_PHASE39_MANIFEST_SHA256:
        continue
    try:
        m = json.loads(p.read_text(encoding='utf-8'))
        if m.get('phase') != EXPECTED_PHASE: continue
        if not all(BASE+s in m.get('artifact_sha256', {}) for s in REQUIRED if s != 'SHAREABLE.json'):
            continue
        manifest_choices.append((p, m, digest))
    except (OSError, UnicodeError, ValueError):
        continue
if not manifest_choices:
    raise RuntimeError('Found Phase 39 manifest filename(s), but none match the pinned Phase 39 '
                       'SHAREABLE.json SHA-256 and expected phase. '
                       'Check that you have the same Phase 39 run on Drive; no substitute was used. '
                       f'Expected manifest SHA-256: {PINNED_PHASE39_MANIFEST_SHA256}')

def folder_distance(a, b):
    ap, bp = a.parent.parts, b.parent.parts
    common = 0
    for x, y in zip(ap, bp):
        if x != y: break
        common += 1
    return len(ap)+len(bp)-2*common

# Hash each likely companion only once across manifests, then choose the matching bytes.
digest_cache = {}
def cached_sha(p):
    if p not in digest_cache: digest_cache[p] = sha(p)
    return digest_cache[p]

def matching_companions(manifest_file, manifest_data):
    chosen, missing = {}, []
    for suffix in REQUIRED[1:]:
        target = manifest_data['artifact_sha256'][BASE+suffix]
        matches = [p for p in candidates[suffix] if cached_sha(p)==target]
        if not matches:
            missing.append(suffix)
        else:
            chosen[suffix] = min(matches, key=lambda p:(folder_distance(p, manifest_file),str(p)))
    return chosen,missing

scored=[]
for manifest_file,manifest_data,digest in manifest_choices:
    chosen,missing = matching_companions(manifest_file,manifest_data)
    distance = sum(folder_distance(x, manifest_file) for x in chosen.values())
    scored.append((len(chosen), -distance, str(manifest_file), manifest_file,manifest_data,digest,chosen,missing))
scored.sort(key=lambda x:(-x[0],-x[1],x[2]))
best = scored[0]
_,_,_,chosen_manifest,manifest,manifest_digest,matching,missing = best
if missing:
    diag = []
    for suffix in missing:
        n = len(candidates[suffix])
        diag.append(BASE+suffix+(' (found '+str(n)+' differently hashed copy/copies)' if n else ' (not found)'))
    raise FileNotFoundError('Verified Phase 39 input set is incomplete on Drive:\n  '+
                            '\n  '.join(diag)+'\nNo manual upload dialog will be opened.')

# Stage the eight approved files to local Colab disk; protects SQLite against Drive FUSE locking.
P = {}
for suffix, source in [('SHAREABLE.json',chosen_manifest)] + list(matching.items()):
    local = IN / (BASE+suffix)
    shutil.copyfile(source,local)
    expected = manifest_digest if suffix=='SHAREABLE.json' else manifest['artifact_sha256'][BASE+suffix]
    assert sha(local)==expected, f'Copy/hash changed during staging: {suffix}'
    P[suffix]=local
verified=[(BASE+s, sha(P[s])) for s in REQUIRED if s!='SHAREABLE.json']
assert len(verified)==7
print('Found verified Phase 39 run in Drive:',chosen_manifest.parent)
print('Staged & independently SHA-256 verified:',len(P),'/',len(REQUIRED),'Phase 39 artifacts')
print('Work files are temporary; all public Phase 40 outputs will be saved on Drive.')


Drive matches by expected artifact: {'SHAREABLE.json': 1, 'SQL_STAGING_COUNTS.csv': 1, 'SOURCE_SQL_RECONCILIATION.csv': 1, 'SOURCE_COHORT_AGREEMENT.csv': 1, 'NEGATIVE_SQL_TESTS.csv': 1, 'ARM_A_READINESS.csv': 1, 'SOURCE_LEDGER_SCHEMA.sql': 1, 'SYNTHETIC_SOURCE_LEDGER.sqlite': 1}
Found verified Phase 39 run in Drive: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs
Staged & independently SHA-256 verified: 8 / 8 Phase 39 artifacts
Work files are temporary; all public Phase 40 outputs will be saved on Drive.


## 2 · Source-ledger integrity and independent reconciliation

Checks operate on the uploaded synthetic ledger in **read-only SQLite mode** and do not rely on the reported Phase 39 pass flags alone. A mismatch or nonempty FK check stops this phase.

In [3]:
# CODE CELL 3/8 — SQLite integrity and report reconciliation
import urllib.parse
ledger=P['SYNTHETIC_SOURCE_LEDGER.sqlite'].resolve()
db=sqlite3.connect('file:'+urllib.parse.quote(str(ledger), safe='/')+'?mode=ro', uri=True)
db.row_factory=sqlite3.Row
assert db.execute('PRAGMA integrity_check').fetchone()[0]=='ok','Database integrity check failed'
assert not db.execute('PRAGMA foreign_key_check').fetchall(),'Database FK check failed'
count_names=['source_person','source_device','source_study','synthetic_procedure_fixture',
             'source_observation','source_observation_provenance']
counts={name:db.execute(f'SELECT COUNT(*) FROM {name}').fetchone()[0] for name in count_names}
assert counts==manifest['results']['sql_table_counts'], ('Ledger count mismatch: observed '+str(counts)+
    '; manifest expected '+str(manifest['results']['sql_table_counts']))
def read_csv(suffix):
    with P[suffix].open(encoding='utf-8-sig',newline='') as f:return list(csv.DictReader(f))
staging=read_csv('SQL_STAGING_COUNTS.csv')
assert len(staging)==len(counts)
for row in staging:
    assert row['table'] in counts
    assert int(row['after_first_load'])==counts[row['table']]
    assert int(row['after_second_load'])==counts[row['table']]
recon=read_csv('SOURCE_SQL_RECONCILIATION.csv')
assert len(recon)==counts['source_observation']
assert all(r['exact_source_sql_match']=='True' and not r['mismatched_fields'] for r in recon)
ledger_obs={r['obs_ref'] for r in db.execute('SELECT obs_ref FROM source_observation')}
assert ledger_obs=={r['observation_ref'] for r in recon}
negative=read_csv('NEGATIVE_SQL_TESTS.csv')
assert len(negative)==12 and all(r['corruption_rejected']=='True' for r in negative)
assert len({r['test'] for r in negative})==12
cohort=read_csv('SOURCE_COHORT_AGREEMENT.csv')
assert len(cohort)==counts['source_person']==19
assert all(r['exact_agreement']=='True' and r['diagnosis_attested']=='False' for r in cohort)
assert len({r['person_ref'] for r in cohort})==19
assert sum(r['source_proxy_eligible']=='True' for r in cohort)==15
assert sum(r['source_proxy_eligible']=='False' for r in cohort)==4
print('Integrity/FK PASS; source counts:',counts)
print('Reconciliation PASS:',len(recon),'observations; negative-control reports:',len(negative),
      '; source-proxy membership: 15 positive / 4 negative (NO attested MCI)')

Integrity/FK PASS; source counts: {'source_person': 19, 'source_device': 19, 'source_study': 38, 'synthetic_procedure_fixture': 38, 'source_observation': 148, 'source_observation_provenance': 148}
Reconciliation PASS: 148 observations; negative-control reports: 12 ; source-proxy membership: 15 positive / 4 negative (NO attested MCI)


## 3 · Independently recompute the *source-defined proxy* cohort

The source proxy requires bilateral hippocampal measurement at baseline and at 12 months, four distinct laterality/timepoint combinations per synthetic person. This is an engineering proxy only; it must never be labeled a diagnosed clinical MCI cohort.

In [4]:
# CODE CELL 4/8 — Recompute source-defined synthetic cohort
sql = """SELECT p.person_ref,
       COUNT(DISTINCT CASE WHEN o.biomarker_code='hippocampus-volume'
                 THEN o.laterality || ':' || o.timepoint END) AS landmark_n
       FROM source_person p LEFT JOIN source_observation o ON p.person_ref=o.person_ref
       GROUP BY p.person_ref ORDER BY p.person_ref"""
computed={r['person_ref']:int(r['landmark_n']) for r in db.execute(sql)}
for r in cohort:
    p=r['person_ref']; n=computed[p]
    assert int(r['source_landmark_count'])==n
    assert int(r['sql_landmark_count'])==n
    assert (n==4)==(r['source_proxy_eligible']=='True')
assert sum(n==4 for n in computed.values())==15
# This audit recomputes an explicit proxy, not an official MI-CDM target query.
print('Source-side synthetic proxy recomputed successfully for',len(computed),'persons.')

Source-side synthetic proxy recomputed successfully for 19 persons.


## 4 · Freeze mapping *requirements*, never fabricate target columns

A proposed **logical** evidence contract based on the research blueprint. Binding to an actual column is forbidden until the exact chosen OMOP CDM and MI-CDM revisions, DDL fingerprints and vocabulary decisions are supplied. `REQUIRES_SCHEMA_BINDING` and `BLOCKED` are deliberate states, not failures of Colab.

**Code cell 5** writes to temporary Colab storage instead of Google Drive. The final cell saves and checksum-verifies the five public-safe outputs in Drive.


In [5]:
# CODE CELL 5/8 — Generate mapping contract locally (Drive-safe)
CONTRACT=[
 ('patient_link','Patient','PERSON','deterministic non-identifying patient mapping','BLOCKED_MISSING_DEMOGRAPHIC_POLICY'),
 ('image_study','ImagingStudy','IMAGE_OCCURRENCE','study and series linkage, acquisition timestamp, modality; no source-attested real DICOM UID','REQUIRES_SCHEMA_BINDING'),
 ('image_measure','Observation','IMAGE_FEATURE / suitable MEASUREMENT if approved','region, laterality, numeric value and units; concept review required','BLOCKED_VOCABULARY_REVIEW'),
 ('extraction_model','Device + Provenance','documented target links or auditable sidecar','algorithm name/version and derivation lineage','REQUIRES_SCHEMA_BINDING'),
 ('reliability','Observation + Provenance','versioned fidelity sidecar','original TRACE state/reason preserved; do not reinterpret as biological truth','REQUIRES_SCHEMA_BINDING'),
 ('longitudinal','ImagingStudy + Observation','image-study / image-feature links','patient-safe baseline-to-12-month pairing','REQUIRES_SCHEMA_BINDING'),
 ('procedures','synthetic-only manufactured Procedure fixture','NO CLINICAL PROCEDURE CLAIM','38 manufactured fixtures are not source-attested procedures','BLOCKED_SOURCE_ABSENT'),
 ('dicom_uid','synthetic study/series identifiers','NO ATTESTED DICOM CLAIM','all current DICOM-style identifiers synthetic','BLOCKED_SOURCE_ABSENT')]
contract_path=STAGE_OUT/'ICHI2027_Phase40_LOGICAL_MAPPING_CONTRACT.csv'
with contract_path.open('w',newline='',encoding='utf-8') as f:
    w=csv.writer(f);w.writerow(['dimension','source_fhir','proposed_logical_target','preservation_invariant','status']);w.writerows(CONTRACT)
assert contract_path.is_file() and contract_path.stat().st_size > 100, 'Mapping contract CSV was not created'
print('Mapping contract generated successfully on local disk:',contract_path.name)

Mapping contract generated successfully on local disk: ICHI2027_Phase40_LOGICAL_MAPPING_CONTRACT.csv


## 5 · Optional private receipts are also discovered from Drive (never exported)

If your Drive has **exactly one** `phase40_private_approvals.json` inside the searched roots, Phase 40 reads it in place and checks that each claimed approval names a real, locally contained, hash-matching evidence file, with `approved=true` and `independently_reviewed=true`. You may explicitly set `PHASE40_PRIVATE_DIR` to the private folder if there are multiple receipts or your evidence is outside the search roots. No receipt or evidence file is written to the public ZIP or GitHub. Missing or unsupported evidence remains **NOT_PROVIDED**, and even a valid receipt does **not** attest that official OMOP/MI-CDM ETL has run.


In [6]:
# CODE CELL 6/8 — Check any optional private receipts
REQUIRED_APPROVALS=['athena_snapshot','binding_review','vocabulary_gaps','demographic_policy',
                    'official_schema_ddl','source_procedures','real_dicom_uids']
receipt={}; receipts_status={key:'NOT_PROVIDED' for key in REQUIRED_APPROVALS}
explicit_private=os.environ.get('PHASE40_PRIVATE_DIR','').strip()
PRIVATE=Path(explicit_private).expanduser().resolve() if explicit_private else None
if not explicit_private:
    if len(private_receipts)==1:
        PRIVATE=private_receipts[0].parent.resolve()
        print('Detected one private receipt on Drive; verifying it locally.')
    elif len(private_receipts)>1:
        print('Multiple private receipts found. Private approvals skipped; set PHASE40_PRIVATE_DIR '
              'to select your intended private evidence folder, without uploading anything.')
if PRIVATE:
    assert PRIVATE.is_dir(),'Configured private evidence directory does not exist'
    receipt_file=PRIVATE/'phase40_private_approvals.json'
    if receipt_file.is_file():
        receipt=json.loads(receipt_file.read_text(encoding='utf-8'))
        for key in REQUIRED_APPROVALS:
            item=receipt.get(key,{})
            relative=item.get('evidence_file','')
            evidence=(PRIVATE/relative).resolve() if isinstance(relative,str) and relative else None
            contained=evidence is not None and evidence.is_relative_to(PRIVATE.resolve())
            valid=(item.get('approved') is True and item.get('independently_reviewed') is True
                   and contained and evidence.is_file() and
                   re.fullmatch('[0-9a-f]{64}',str(item.get('sha256',''))) is not None
                   and sha(evidence)==item['sha256'])
            receipts_status[key]='LOCAL_HASH_AND_ATTESTATION_VERIFIED' if valid else 'MISSING_OR_INVALID'
print('PRIVATE gate receipts (statuses only; no private evidence copied to outputs):')
for k,v in receipts_status.items():print('  ',k,':',v)
# Later official target-syntactic loader must independently verify schema semantics/record-level adjudication.


PRIVATE gate receipts (statuses only; no private evidence copied to outputs):
   athena_snapshot : NOT_PROVIDED
   binding_review : NOT_PROVIDED
   vocabulary_gaps : NOT_PROVIDED
   demographic_policy : NOT_PROVIDED
   official_schema_ddl : NOT_PROVIDED
   source_procedures : NOT_PROVIDED
   real_dicom_uids : NOT_PROVIDED


## 6 · Ten original readiness gates + evidence-gap summary

Preserves all 10 Phase 39 blockers. The private approval receipt can establish *readiness to perform additional checks* for some gates, but **cannot** prove that an official load, unchanged generic TFL, external challenge, or authorized MRI run happened.

In [7]:
# CODE CELL 7/8 — Compile Arm A evidence-gate status locally
prior=read_csv('ARM_A_READINESS.csv')
assert len(prior)==10 and all(r['passed']=='False' for r in prior)
expected_gates={
 'authorized_frozen_athena_with_valid_manifest':'athena_snapshot',
 '19_binding_human_adjudication':'binding_review',
 'original_36_vocabulary_gap_instances_resolved':'vocabulary_gaps',
 'source_supported_clinical_procedures':'source_procedures',
 'real_dicom_uid_source_attested':'real_dicom_uids',
 'missing_gender_and_birthdate_policy_independently_approved':'demographic_policy'
}
assert set(expected_gates).issubset({r['gate'] for r in prior})
readiness=[]
for r in prior:
    receipt_key=expected_gates.get(r['gate'])
    evidence=receipts_status.get(receipt_key,'REQUIRES_FUTURE_EXECUTION') if receipt_key else 'REQUIRES_FUTURE_EXECUTION'
    readiness.append({'gate':r['gate'],'phase39_passed':r['passed'],
                     'phase40_evidence_receipt':evidence,
                     'official_validation_passed':'False',
                     'next_action':r['action']})
# Hard blocks, even if independently reviewed: this phase does not execute official ETL or clinical attestation.
armA_status='BLOCKED_OFFICIAL_ARM_A_NOT_EXECUTED'
out_ready=STAGE_OUT/'ICHI2027_Phase40_ARM_A_EVIDENCE_GATES.csv'
with out_ready.open('w',newline='',encoding='utf-8') as f:
    w=csv.DictWriter(f,fieldnames=list(readiness[0]));w.writeheader();w.writerows(readiness)
print('Arm A:',armA_status,'; gates remaining to establish:',len(readiness))

Arm A: BLOCKED_OFFICIAL_ARM_A_NOT_EXECUTED ; gates remaining to establish: 10


## 7 · Save the public-safe outputs directly to Google Drive

After building all outputs locally, Phase 40 writes `SHAREABLE.json`, `ARM_A_EVIDENCE_GATES.csv`, `LOGICAL_MAPPING_CONTRACT.csv`, `README.txt`, and a public-safe ZIP to the dated Phase 40 Drive output folder printed in the first cell. **No uploads and no downloads are needed.** All inputs remain in Drive, and the temporary SQLite copy is used only for source-side verification.


In [8]:
# CODE CELL 8/8 — Package results and save verified copies to Drive
summary={
 'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
 'phase':'40_source_evidence_gate_and_logical_mapping_contract',
 'run_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'scope':'PUBLIC_SYNTHETIC_SOURCE_SIDE_ONLY',
 'phase39_input_manifest_sha256':sha(P['SHAREABLE.json']),
 'phase39_companion_sha256_verified':len(verified),
 'source_db_integrity_check':'PASS', 'source_db_fk_check':'PASS',
 'synthetic_people':len(computed),'synthetic_studies':counts['source_study'],
 'synthetic_observations':counts['source_observation'],
 'synthetic_cohort_proxy_positive':15,'synthetic_cohort_proxy_negative':4,
 'clinical_mci_attested':False,
 'public_phase39_negative_control_report_checks':len(negative),
 'synthetic_source_side_reconciliation':'PASS',
 'logical_mapping_contract_rows':len(CONTRACT),
 'private_evidence_receipt_status_counts':dict(Counter(receipts_status.values())),
 'official_omop_etl_executed':False,'official_micdm_load_executed':False,
 'armA_status':armA_status,'real_mri_rows_processed':0,
 'full_four_arm_evaluation':False,
 'warning':'Logical contract is not pinned official schema DDL or a completed mapping implementation.'
}
manifest_path=STAGE_OUT/'ICHI2027_Phase40_SHAREABLE.json'
manifest_path.write_text(json.dumps(summary,indent=2)+'\n',encoding='utf-8')
README=STAGE_OUT/'ICHI2027_Phase40_README.txt'
README.write_text('Phase 40 public source-side verification only.\n'+
                  'Files: SHAREABLE.json; ARM_A_EVIDENCE_GATES.csv; LOGICAL_MAPPING_CONTRACT.csv.\n'+
                  'NO clinical MCI, Athena concept-ID decisions, official OMOP/MI-CDM loading, or real MRI validation claimed.\n'+
                  'Do not upload private receipt or linked evidence to GitHub.\n',encoding='utf-8')
# Guard against accidental private-path/receipt leakage in output.
public_files=[manifest_path,out_ready,contract_path,README]
for p in public_files:
    body=p.read_text(encoding='utf-8')
    if PRIVATE: assert str(PRIVATE.resolve()) not in body, 'Private path leaked into public output'
print('Public outputs:',[(p.name,sha(p)) for p in public_files])
import zipfile
zpath=STAGE_OUT/'ICHI2027_Phase40_PUBLIC_OUTPUTS.zip'
with zipfile.ZipFile(zpath,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in public_files:z.write(p,p.name)
print('Local ZIP created:', zpath.name)
# Copy finalized files, never create them directly on Google Drive's FUSE mount.
# Verify read-back checksums so success cannot be claimed for truncated writes.
import time
saved = []
for source_file in public_files + [zpath]:
    destination = OUT / source_file.name
    expected_digest = sha(source_file)
    last_error = None
    for attempt in range(3):
        try:
            shutil.copyfile(source_file, destination)
            actual_digest = sha(destination)
            if actual_digest != expected_digest:
                raise OSError('Drive read-back checksum mismatch for '+source_file.name)
            saved.append(destination)
            last_error = None
            break
        except OSError as exc:
            last_error = exc
            print('Drive write retry',attempt+1,'/3 for',source_file.name,':',str(exc))
            time.sleep(1 + attempt)
    if last_error is not None:
        raise RuntimeError('All Phase 40 outputs were generated on local Colab disk, but Google Drive '
                           'could not save '+source_file.name+'. Check storage quota, permissions, '
                           'and the Drive mount; then rerun only this final code cell. '
                           'Local backup directory: '+str(STAGE_OUT)) from last_error
assert len(saved) == 5 and all(p.is_file() for p in saved)
print('SUCCESS: all five outputs saved to Drive and rehashed:',OUT)
print('Public ZIP:', OUT / zpath.name)


Public outputs: [('ICHI2027_Phase40_SHAREABLE.json', 'd52d550911fa784935c9dbc2956fa623f60432c9814bf8714ccca7c8a25205d2'), ('ICHI2027_Phase40_ARM_A_EVIDENCE_GATES.csv', '4c1febebd83ee969f9102499ddd09fba6542092c202fcfd7d98015797f1f454b'), ('ICHI2027_Phase40_LOGICAL_MAPPING_CONTRACT.csv', 'd80e7ede309fedca3e95341b299ae4756c9a6796ea2148c4abc0479f8ee235da'), ('ICHI2027_Phase40_README.txt', '46217535221fe5d0d170c630ee70da67b778cb6a50e21ccd9578f3031bcf9186')]
Local ZIP created: ICHI2027_Phase40_PUBLIC_OUTPUTS.zip
SUCCESS: all five outputs saved to Drive and rehashed: /content/drive/MyDrive/ICHI2027/Phase40/run_20260929T054826_100243Z
Public ZIP: /content/drive/MyDrive/ICHI2027/Phase40/run_20260929T054826_100243Z/ICHI2027_Phase40_PUBLIC_OUTPUTS.zip


## Next gate (Phase 41; do not skip)

The future **official Arm A target loader** needs an authorized frozen Athena snapshot and independent 19-binding review, resolutions for all 36 original vocabulary-gap instances, exact approved OMOP/MI-CDM DDL, approved demographic handling, and honest provenance decisions where the original source has **no attested clinical Procedure or real DICOM UID**. Until those exist and official code actually runs, Phase 40 correctly reports Arm A **blocked**, even if all synthetic source checks pass. Do not re-label the 19 synthetic participants as a real ADNI/MCI cohort.

**GitHub:** only the public-safe Phase 40 ZIP contents, if you choose to commit them. Keep any Athena data, adjudicator details, participant identifiers, private receipt, and associated evidence outside the public repository.
